[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vonKleve/csci-e222-final-project/blob/master/ner/03-medgemma-lora.ipynb)

# Fine Tuning MedGemma-4B for Medical NER

## Description
This notebook fine-tunes `google/medgemma-4b-it` on the `Ben10x/MedMentions-MTI881-NER` dataset.

Unlike BERT models which handle Named Entity Recognition (NER) as a *token classification* task, Large Language Models handle NER as a **text-to-text task**. The model is prompted with the text and taught to generate a string that pairs words with their respective entity tags.

I use **Unsloth** and **QLoRA (Quantized Low-Rank Adaptation)** to make training this model feasible.

In [1]:
%%capture
!pip install --upgrade transformers bitsandbytes accelerate trl unsloth datasets

In [2]:
import os
# Suppress all tqdm/HuggingFace progress bars before any library is imported
os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"

In [3]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import notebook_login


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [4]:
import os
os.environ["TQDM_DISABLE"] = "1"

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from datasets import disable_progress_bars
disable_progress_bars()

from huggingface_hub.utils import disable_progress_bars as hub_disable_progress_bars
hub_disable_progress_bars()

In [ ]:
PUSH_TO_HUB = True  # Set to False to skip login and hub push

if PUSH_TO_HUB:
    notebook_login()

In [6]:
import time
notebook_start_time = time.time()


## 1. Hyperparameters

* **Load in 4-bit (`True`)**: reduces the RAM required to load the model (from ~16GB to ~4GB) without significant performance degradation.
* **LoRA Rank (`r=16`) & Alpha (`16`)**: rank 16 provides a good balance between expressive capacity and memory efficiency. Setting Alpha equal to Rank is a standard best practice for stable scaling in QLoRA.
* **Learning Rate (`2e-4`)**: LLMs fine-tuned with LoRA require higher learning rates than full-parameter fine-tuning (which normally uses `2e-5`), for LoRA matrices to meaningfully influence the target matrices.
* **Batch Size (`4`) & Gradient Accumulation (`4`)**: an effective batch size of 16.
* **Generative NER Format**: need to append the `EOS` (End of Sequence) token to the end of target responses during training so the model learns when to stop generating.

In [7]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

class Config:
    MODEL_ID = "google/medgemma-4b-it"
    DATASET_ID = "Ben10x/MedMentions-MTI881-NER"
    HUB_REPO_ID = "alexd063/medgemma-finetuned-medmentions-ner"

    MAX_SEQ_LENGTH = 512
    LOAD_IN_4BIT = True

    # LoRA Config
    LORA_R = 16
    LORA_ALPHA = 16
    LORA_DROPOUT = 0

    # Training Config
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 4
    LEARNING_RATE = 2e-4
    # EPOCHS = 1               # either EPOCHS or MAX_STEPS (mutually exclusive)
    MAX_STEPS = 300        # -1/remove for full epoch
    WARMUP_STEPS = 5
    WEIGHT_DECAY = 0.01

config = Config()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.MODEL_ID,
    max_seq_length=config.MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=config.LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=config.LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"], # attention
    lora_alpha=config.LORA_ALPHA,
    lora_dropout=config.LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

==((====))==  Unsloth 2025.11.1: Fast Gemma3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


## 2. Prompt Formatting for NER
Map token tags into instruction format.
Ensure that `eos` token is appended so that the model learns when to stop predicting.

In [8]:
dataset = load_dataset(config.DATASET_ID)

tokenizer.pad_token = tokenizer.eos_token # (gemma uses eos token for padding)

def format_instruction(sample):
    instruction = "Extract medical entities from the following text and label each token with its NER tag."

    input_text = " ".join(sample["tokens"]) if "tokens" in sample else sample["text"]

    # tokens and tags in a string: "Token1: Tag1, Token2: Tag2"
    if "tokens" in sample and "ner_tags" in sample:
        pairs = [f"{token}: {tag}" for token, tag in zip(sample["tokens"], sample["ner_tags"])]
        output_text = ", ".join(pairs)
    else:
        output_text = sample.get("output", "")

    prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n{output_text}{tokenizer.eos_token}"
    )

    return {"text": prompt}

print("Formatting datasets...")
train_dataset = dataset["train"].map(format_instruction)
val_dataset = dataset["validation"].map(format_instruction)

print("Example Prompt:")
print(train_dataset[0]["text"])

Formatting datasets...
Example Prompt:
### Instruction:
Extract medical entities from the following text and label each token with its NER tag.

### Input:
High levels of Lp(a ) may be particularly important in the pathogenesis of CAD in AAs .

### Response:
High: O, levels: O, of: O, Lp(a: B-T103, ): I-T103, may: O, be: O, particularly: O, important: O, in: O, the: O, pathogenesis: B-T038, of: O, CAD: B-T038, in: O, AAs: B-T098, .: O<end_of_turn>


## 3. Training
Using TRL's `SFTTrainer`: `eval_strategy` and `logging_steps` to track validation and training loss.

In [ ]:
from transformers import EarlyStoppingCallback

training_args = SFTConfig(
    output_dir="./medgemma-finetuned",
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    gradient_accumulation_steps=config.GRAD_ACCUM_STEPS,
    # learning rate
    warmup_steps=config.WARMUP_STEPS,
    max_steps=config.MAX_STEPS,
    # num_train_epochs=config.EPOCHS,
    learning_rate=config.LEARNING_RATE,
    # optimization
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    # eval strategy
    eval_strategy="steps",
    eval_steps=20,
    save_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # other
    optim="adamw_8bit",
    weight_decay=config.WEIGHT_DECAY,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=config.MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer_stats = trainer.train()

model.save_pretrained("medgemma-final-lora")
tokenizer.save_pretrained("medgemma-final-lora")

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
{'loss': 0.3829, 'grad_norm': 0.10365322977304459, 'learning_rate': 0.00019728813559322035, 'epoch': 0.006837606837606838}
{'loss': 0.2535, 'grad_norm': 0.07465249300003052, 'learning_rate': 0.0001905084745762712, 'epoch': 0.013675213675213675}
{'eval_loss': 0.8485057950019836, 'eval_runtime': 162.0598, 'eval_samples_per_second': 18.043, 'eval_steps_per_second': 4.511, 'epoch': 0.013675213675213675}
{'loss': 0.1934, 'grad_norm': 0.06685192883014679, 'learning_rate': 0.00018372881355932204, 'epoch': 0.020512820512820513}
{'loss': 0.1672, 'grad_norm': 0.05837450176477432, 'learning_rate': 0.0001769491525423729, 'epoch': 0.02735042735042735}


## 4. Evaluation
Generative NER cannot be evaluated easily using traditional exact match metrics due to it's nature. Without resorting to writing complex parsing algorithm, I instead monitor training, validation losses to ensure convergence and then perform visual assessment how well the model translated raw text into tagged entities.

In [ ]:
history = trainer.state.log_history

train_loss = [x['loss'] for x in history if 'loss' in x]
train_steps = [x['step'] for x in history if 'loss' in x]

val_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]
val_steps = [x['step'] for x in history if 'eval_loss' in x]

# loss function
plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_loss, label='Training Loss', color='blue', linewidth=2)

if len(val_loss) > 0:
    plt.plot(val_steps, val_loss, label='Validation Loss', color='orange', marker='o', linewidth=2)

plt.title('MedGemma-4B Generative NER Loss', fontsize=14)
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('CrossEntropy Loss', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def evaluate_qualitative(model, tokenizer, dataset, num_samples=3):
    FastLanguageModel.for_inference(model) # optimization for speed

    for i in range(num_samples):
        sample = dataset[i]

        # split text to provide model with everything except the answer
        prompt = sample['text'].split("### Response:")[0] + "### Response:\n"
        ground_truth = sample['text'].split("### Response:")[-1].replace(tokenizer.eos_token, "").strip()

        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                use_cache=True,
                temperature=0.1 # low temp for strict factual extraction
            )

        # decoding & parsing the response
        prediction = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        response_only = prediction.split("### Response:")[-1].strip()

        print(f"\n{'='*50}")
        print(f" SAMPLE {i+1} ")
        print(f"{'='*50}")
        print(f"INPUT:\n{prompt.split('### Input:')[1].split('### Response:')[0].strip()}\n")
        print(f"EXPECTED:\n{ground_truth}\n")
        print(f"PREDICTED:\n{response_only}")

evaluate_qualitative(model, tokenizer, val_dataset)

In [ ]:
notebook_end_time = time.time()
elapsed = notebook_end_time - notebook_start_time
hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Total notebook execution time: {int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}")


In [ ]:
if PUSH_TO_HUB:
    model.push_to_hub(config.HUB_REPO_ID, token=True)
    tokenizer.push_to_hub(config.HUB_REPO_ID, token=True)